- Splitting a string into a list of words is known as tokenization.
- One of the most popular tokenization
comes from NLTK (Natural Language Tool Kit).

In [8]:
# !pip install nltk
import nltk
# nltk.download('punkt')
# nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize
sentence="hi,how are you?"


In [9]:
sentence.split()

['hi,how', 'are', 'you?']

In [10]:
word_tokenize(sentence)

['hi', ',', 'how', 'are', 'you', '?']

One of the basic models that you should always try with a classification problem in
NLP is 
### - bag of words.

In bag of words, we create a huge sparse matrix that stores
counts of all the words in our corpus (corpus = all the documents = all the
sentences)

- For this, we will use CountVectorizer from scikit-learn.

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

# create a corpus of sentences
corpus = [
"hello, how are you?",
"im getting bored at home. And you? What do you think?",
"did you know about counts",
"let's see if this works!",
"YES!!!!"
]

# initialize CountVectorizer
ctv=CountVectorizer()

ctv.fit(corpus)

corpus_transformed=ctv.transform(corpus)

In [12]:
corpus_transformed

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 25 stored elements and shape (5, 23)>

In [13]:
print(corpus_transformed)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 25 stored elements and shape (5, 23)>
  Coords	Values
  (0, 2)	1
  (0, 9)	1
  (0, 11)	1
  (0, 22)	1
  (1, 1)	1
  (1, 3)	1
  (1, 4)	1
  (1, 7)	1
  (1, 8)	1
  (1, 10)	1
  (1, 13)	1
  (1, 17)	1
  (1, 19)	1
  (1, 22)	2
  (2, 0)	1
  (2, 5)	1
  (2, 6)	1
  (2, 14)	1
  (2, 22)	1
  (3, 12)	1
  (3, 15)	1
  (3, 16)	1
  (3, 18)	1
  (3, 20)	1
  (4, 21)	1


In [14]:
print(ctv.vocabulary_)

{'hello': 9, 'how': 11, 'are': 2, 'you': 22, 'im': 13, 'getting': 8, 'bored': 4, 'at': 3, 'home': 10, 'and': 1, 'what': 19, 'do': 7, 'think': 17, 'did': 6, 'know': 14, 'about': 0, 'counts': 5, 'let': 15, 'see': 16, 'if': 12, 'this': 18, 'works': 20, 'yes': 21}


In [17]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import word_tokenize

# create a corpus of sentences
corpus=[
"hello, how are you?",
"im getting bored at home. And you? What do you think?",
"did you know about counts",
"let's see if this works!",
"YES!!!!"
]
# initialize CountVectorizer with word_tokenize from ntlk
# as the tokenizer

ctv=CountVectorizer(tokenizer=word_tokenize,token_pattern=None)

# fit the vectorizer on corpus
ctv.fit(corpus)

corpus_transformed=ctv.transform(corpus)

print(ctv.vocabulary_)

{'hello': 14, ',': 2, 'how': 16, 'are': 7, 'you': 27, '?': 4, 'im': 18, 'getting': 13, 'bored': 9, 'at': 8, 'home': 15, '.': 3, 'and': 6, 'what': 24, 'do': 12, 'think': 22, 'did': 11, 'know': 19, 'about': 5, 'counts': 10, 'let': 20, "'s": 1, 'see': 21, 'if': 17, 'this': 23, 'works': 25, '!': 0, 'yes': 26}


In [19]:
import os 
os.getcwd()

'/Users/akhichoudhary/STATS/Thinkstats/Thinkstats2_exercise/Thinkstats/Approach any problem with ml'

### Building Models
- Dataset -IMDB movie review dataset

In [21]:
import pandas as pd
df = pd.read_csv("/Users/akhichoudhary/STATS/Thinkstats/Thinkstats2_exercise/Thinkstats/Approach any problem with ml/mnist_classifier/input/All_dataset/imdb.csv")

In [22]:
df.head(5)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


### CountVectorizer - Logistic Regression

In [25]:
import pandas as pd
from nltk.tokenize import word_tokenize
from sklearn import linear_model
from sklearn import metrics
from sklearn import model_selection
from sklearn.feature_extraction.text import CountVectorizer
if __name__ == "__main__":
    # read the training data
    df = pd.read_csv("/Users/akhichoudhary/STATS/Thinkstats/Thinkstats2_exercise/Thinkstats/Approach any problem with ml/mnist_classifier/input/All_dataset/imdb.csv")
    # map positive to 1 and negative to 0
    df.sentiment = df.sentiment.apply(lambda x: 1 if x == "positive" else 0)
    # we create a new column called kfold and fill it with -1
    df["kfold"] = -1
    # the next step is to randomize the rows of the data
    df = df.sample(frac=1).reset_index(drop=True)
    # fetch labels
    y = df.sentiment.values
    # initiate the kfold class from model_selection module
    kf = model_selection.StratifiedKFold(n_splits=5)
    # fill the new kfold column
    for f, (t_, v_) in enumerate(kf.split(X=df, y=y)):
        df.loc[v_, 'kfold'] = f
        # we go over the folds created
    
    for fold_ in range(5):
        # temporary dataframes for train and test
        train_df = df[df.kfold != fold_].reset_index(drop=True)
        test_df = df[df.kfold == fold_].reset_index(drop=True)
        # initialize CountVectorizer with NLTK's word_tokenize
        # function as tokenizer
        count_vec = CountVectorizer(tokenizer=word_tokenize,token_pattern=None)
        # fit count_vec on training data reviews
        count_vec.fit(train_df.review)
        # transform training and validation data reviews
        xtrain = count_vec.transform(train_df.review)
        xtest = count_vec.transform(test_df.review)
        # initialize logistic regression model
        model = linear_model.LogisticRegression()
        # fit the model on training data reviews and sentiment
        model.fit(xtrain, train_df.sentiment)
        # make predictions on test data
        # threshold for predictions is 0.5
        preds = model.predict(xtest)
        # calculate accuracy
        accuracy = metrics.accuracy_score(test_df.sentiment, preds)
        print(f"Fold: {fold_}")
        print(f"Accuracy = {accuracy}")
        print("")


/Users/akhichoudhary/miniconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold: 0
Accuracy = 0.8889



/Users/akhichoudhary/miniconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold: 1
Accuracy = 0.8915



/Users/akhichoudhary/miniconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold: 2
Accuracy = 0.892



/Users/akhichoudhary/miniconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Fold: 3
Accuracy = 0.8919

Fold: 4
Accuracy = 0.8895



/Users/akhichoudhary/miniconda3/envs/ml/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Naive Bayes 

In [24]:
import pandas as pd

from sklearn import metrics
from sklearn import naive_bayes
from nltk.tokenize import word_tokenize
from sklearn import model_selection
from sklearn.feature_extraction.text import CountVectorizer


if __name__=="__main__":
    df = pd.read_csv("/Users/akhichoudhary/STATS/Thinkstats/Thinkstats2_exercise/Thinkstats/Approach any problem with ml/mnist_classifier/input/All_dataset/imdb.csv")
    df.sentiment=df.sentiment.apply(lambda x: 1 if x=='positive' else 0)

    df['kfold']=-1

    df=df.sample(frac=1).reset_index(drop=True)

    y=df.sentiment.values

    kf=model_selection.StratifiedKFold(n_splits=5)

    for f_,(t_,v_) in enumerate(kf.split(X=df,y=y)):
        df.loc[v_,'kfold']=f_


    for fold_ in range(5):
        train_df=df[df.kfold!=fold_].reset_index(drop=True)

        test_df=df[df.kfold==fold_].reset_index(drop=True)

        count_vec=CountVectorizer(
        tokenizer=word_tokenize,
        token_pattern=None)

        count_vec.fit(train_df.review)

        xtrain=count_vec.transform(train_df.review)
        xtest=count_vec.transform(test_df.review)

        model=naive_bayes.MultinomialNB()

        model.fit(xtrain,train_df.sentiment)

        preds=model.predict(xtest)

        accuracy=metrics.accuracy_score(test_df.sentiment,preds)

        print(f"Fold: {fold_}")
        print(f"Accuracy = {accuracy}")
        print("")
# ══════════════════

Fold: 0
Accuracy = 0.8417

Fold: 1
Accuracy = 0.8361

Fold: 2
Accuracy = 0.8423

Fold: 3
Accuracy = 0.8532

Fold: 4
Accuracy = 0.8432



-  The score is low but it's superfast

### TF-IDF
- TF > Term frequencies
- IDF > Inverse document Frequency

- formulae for TF and IDF.
    
          Number of times a term t appears in a document
TF(t) =   ──────────────────────────────────
          Total number of terms in the document

                  Total number of documents
IDF(t) = LOG(  ──────────────────────────────────   )
              Number of documents with term t in it

And TF-IDF for a term t is defined as:
    
- TF-IDF(t) = TF(t) * IDF(t)
Similar to CountVectorizer in scikit-learn, we have TfidfVectorizer. Let’s try using
it the same way we used CountVectorizer.

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize

# create a corpus of sentences
corpus=[
"hello, how are you?",
"im getting bored at home. And you? What do you think?",
"did you know about counts",
"let's see if this works!",
"YES!!!!"
]
# initialize CountVectorizer with word_tokenize from ntlk
# as the tokenizer

ctv=TfidfVectorizer(tokenizer=word_tokenize,token_pattern=None)

# fit the vectorizer on corpus
ctv.fit(corpus)

corpus_transformed=ctv.transform(corpus)

print(ctv.vocabulary_)

{'hello': 14, ',': 2, 'how': 16, 'are': 7, 'you': 27, '?': 4, 'im': 18, 'getting': 13, 'bored': 9, 'at': 8, 'home': 15, '.': 3, 'and': 6, 'what': 24, 'do': 12, 'think': 22, 'did': 11, 'know': 19, 'about': 5, 'counts': 10, 'let': 20, "'s": 1, 'see': 21, 'if': 17, 'this': 23, 'works': 25, '!': 0, 'yes': 26}


In [27]:
from sklearn.feature_extraction.text import TfidVectorizer


ImportError: cannot import name 'TfidVectorizer' from 'sklearn.feature_extraction.text' (/Users/akhichoudhary/miniconda3/envs/ml/lib/python3.10/site-packages/sklearn/feature_extraction/text.py)